# Ergebnisse – Experiment-Summary
- Pro Experiment & Datensatz eine Tabelle (mit `n_runs` bei Exp 1/2/4); Bar-Chart bei Exp 1/2/4, bei Exp 3 SHAP-Tabelle + Bar-Charts (Top-Features und Text vs. numerisch)
- Quelle: alle MLflow-Runs aus `./mlruns`; fehlende/leere Experimente werden übersprungen

In [ ]:
import os
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = "result_tables"
os.makedirs(RESULT_DIR, exist_ok=True)

## Runs laden
- Nur vorhandene Experimente aus `./mlruns`; leere/fehlende werden übersprungen

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")

experiments = [f"{ds}_experiment_{i}" for ds in ["fake_jobs", "airbnb_paris"] for i in [1, 2, 3, 4, 5]]

parts = []
for name in experiments:
    exp = mlflow.get_experiment_by_name(name)
    if exp is None:
        continue
    runs = mlflow.search_runs([exp.experiment_id])
    if len(runs) == 0:
        continue
    runs["experiment"] = name
    parts.append(runs)

df = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(f"{len(df)} Runs aus {len(parts)} Experimenten geladen")

## Aufbereiten
- Datensatz / Experiment-Nr. / Modellname ableiten; Metrik-Spalten vereinheitlichen

In [ ]:
datasets = ["fake_jobs", "airbnb_paris"]

if len(df):
    df["exp_num"] = df["experiment"].str.split("_").str[-1].astype(int)
    df["dataset"] = df["experiment"].str.rsplit("_experiment_", n=1).str[0]
    df["model"] = df["tags.mlflow.runName"] if "tags.mlflow.runName" in df.columns else df["run_id"]
    df = df[df["model"] != "contexttab_fixed"]  # veralteten Run ausschließen
    df = df.rename(columns={"metrics.average_precision": "AP", "metrics.auprc": "AUPRC", "metrics.auc_roc": "AUROC", "metrics.runtime_s": "runtime_s"})
    for col in ["AP", "AUPRC", "AUROC", "runtime_s"]:
        if col not in df.columns:
            df[col] = np.nan
    print("Datensätze:", df["dataset"].unique().tolist(), "| Experimente:", sorted(df["exp_num"].unique()))

## Experiment 1 – Unsupervised Outlier Detection
- TFMs (TabPFN unsup., AnoLLM, FoMo-OD) vs. PyOD-Baselines; AP, AUPRC & AUROC je Modell
- Tabelle je Datensatz; je 1 kombinierter Chart für AP, AUPRC, AUROC und Runtime (beide Datensätze, Modelle alphabetisch)

In [ ]:
sub = df[df["exp_num"] == 1] if len(df) else df
if not len(sub):
    print("keine Exp-1-Runs")
else:
    # Tabelle je Datensatz
    for ds in datasets:
        s = sub[sub["dataset"] == ds]
        if not len(s):
            continue
        g = s.groupby("model")
        tab = g[["AP", "AUPRC", "AUROC", "runtime_s"]].mean()
        tab["n_runs"] = g.size()
        tab = tab.sort_values("AP", ascending=False).round(4)
        tab.to_csv(f"{RESULT_DIR}/exp1_{ds}.csv")
        print(f"=== Experiment 1 – {ds} ===")
        display(tab)

    # je 1 Chart für AP, AUROC, Runtime – beide Datensätze, Modelle alphabetisch
    for metric, label in [("AP", "Average Precision"), ("AUPRC", "AUPRC"), ("AUROC", "AUROC"), ("runtime_s", "Runtime (s)")]:
        piv = sub.pivot_table(index="model", columns="dataset", values=metric, aggfunc="mean").sort_index()
        ax = piv.plot.bar(figsize=(10, 4), rot=30)
        ax.set_title(f"Experiment 1 – {label} (beide Datensätze)")
        ax.set_ylabel(label)
        ax.set_xlabel("")
        ax.legend(title="Datensatz")
        plt.tight_layout()
        plt.show()

## Experiment 2 – Enhanced Baseline-Modelle
- Baselines auf cleaned vs. semantisch vs. enhanced vs. enhanced+semantisch (AP, AUPRC & AUROC je Detektor)
- Je Datensatz je 1 Chart für AP, AUROC und AUPRC
- Hinweis: enhanced ist semi-supervised (Label-Leakage), nicht direkt mit unsupervised vergleichbar

In [ ]:
for ds in datasets:
    sub = df[(df["exp_num"] == 2) & (df["dataset"] == ds)] if len(df) else df
    if not len(sub):
        print(f"{ds}: keine Exp-2-Runs")
        continue
    piv = sub.pivot_table(index="params.detector", columns="params.representation", values="AP", aggfunc=["mean", "size"]).round(4)
    piv = piv.rename(columns={"mean": "AP", "size": "n_runs"}, level=0)
    piv.to_csv(f"{RESULT_DIR}/exp2_{ds}.csv")
    print(f"=== Experiment 2 – {ds} (AP & n_runs je Detektor × Repräsentation) ===")
    display(piv)
    ax = piv["AP"].plot.bar(figsize=(10, 4), rot=0)
    ax.set_title(f"Experiment 2 – {ds}: AP je Repräsentation")
    ax.set_ylabel("Average Precision")
    ax.set_xlabel("Detektor")
    ax.legend(title="Repräsentation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    # AUROC je Detektor × Repräsentation
    piv_auc = sub.pivot_table(index="params.detector", columns="params.representation", values="AUROC", aggfunc="mean").round(4)
    ax = piv_auc.plot.bar(figsize=(10, 4), rot=0)
    ax.set_title(f"Experiment 2 – {ds}: AUROC je Repräsentation")
    ax.set_ylabel("AUROC")
    ax.set_xlabel("Detektor")
    ax.legend(title="Repräsentation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

    # AUPRC je Detektor × Repräsentation
    piv_auprc = sub.pivot_table(index="params.detector", columns="params.representation", values="AUPRC", aggfunc="mean").round(4)
    ax = piv_auprc.plot.bar(figsize=(10, 4), rot=0)
    ax.set_title(f"Experiment 2 – {ds}: AUPRC je Repräsentation")
    ax.set_ylabel("AUPRC")
    ax.set_xlabel("Detektor")
    ax.legend(title="Repräsentation", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

## Experiment 3 – Semantische Relevanz (SHAP)
- ConTextTab-SHAP: wichtigste Features je Datensatz (nur Tabelle)

In [ ]:
for ds in datasets:
    sub = df[(df["exp_num"] == 3) & (df["dataset"] == ds)] if len(df) else df
    if not len(sub):
        continue
    rel_cols = [c for c in sub.columns if c.startswith("metrics.") and not c.endswith("_total") and sub[c].notna().any()]
    rel = sub.iloc[0][rel_cols].dropna().astype(float)
    rel.index = [c.replace("metrics.", "") for c in rel.index]
    rel_type = pd.Series(["text" if i.startswith("text_") else "numeric" for i in rel.index], index=rel.index)
    top = rel.sort_values(ascending=False).head(20).round(4).to_frame("mean_abs_shap")
    top["type"] = [rel_type[i] for i in top.index]
    top.index = [i.replace("text_", "").replace("num_", "") for i in top.index]
    top.to_csv(f"{RESULT_DIR}/exp3_{ds}.csv")
    print(f"=== Experiment 3 – {ds}: Top 20 Features (Text + numerisch) ===")
    display(top)

    # bar chart: top features colored by type
    colors = {"text": "tab:orange", "numeric": "tab:blue"}
    ax = top["mean_abs_shap"][::-1].plot.barh(figsize=(8, 6), color=[colors[t] for t in top["type"][::-1]])
    ax.set_title(f"Experiment 3 – {ds}: Top 20 SHAP-Features")
    ax.set_xlabel("mean(|SHAP|)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[t]) for t in ["text", "numeric"]]
    ax.legend(handles, ["text", "numeric"], title="Typ")
    plt.tight_layout()
    plt.show()

    # second chart: total SHAP contribution text vs numeric
    by_type = rel.groupby(rel_type).sum().reindex(["text", "numeric"]).fillna(0).round(4)
    ax = by_type.plot.bar(figsize=(4, 4), rot=0, color=[colors[t] for t in by_type.index])
    ax.set_title(f"Experiment 3 – {ds}: SHAP Text vs. numerisch (Summe)")
    ax.set_ylabel("Summe mean(|SHAP|)")
    ax.set_xlabel("")
    plt.tight_layout()
    plt.show()

## Experiment 4 – TFMs zur binären Klassifikation
- ConTextTab vs. TabPFN-Klassifikation; AP, AUPRC & AUROC je Modell
- Je Modell zwei `distribution`-Varianten (`original`, `balanced_1to4`) als eigene Runs (`*_original` / `*_balanced_1to4`)
- Tabelle je Datensatz; je 1 kombinierter Chart für AP, AUPRC und AUROC (beide Datensätze, Modelle alphabetisch)

In [ ]:
sub = df[df["exp_num"] == 4] if len(df) else df
if not len(sub):
    print("keine Exp-4-Runs")
else:
    # Tabelle je Datensatz
    for ds in datasets:
        s = sub[sub["dataset"] == ds]
        if not len(s):
            continue
        g = s.groupby("model")
        tab = g[["AP", "AUPRC", "AUROC", "runtime_s"]].mean()
        tab["n_runs"] = g.size()
        tab = tab.sort_values("AP", ascending=False).round(4)
        tab.to_csv(f"{RESULT_DIR}/exp4_{ds}.csv")
        print(f"=== Experiment 4 – {ds} ===")
        display(tab)

    # AP- & AUROC-Chart – beide Datensätze, Modelle alphabetisch
    for metric, label in [("AP", "Average Precision"), ("AUPRC", "AUPRC"), ("AUROC", "AUROC")]:
        piv = sub.pivot_table(index="model", columns="dataset", values=metric, aggfunc="mean").sort_index()
        ax = piv.plot.bar(figsize=(8, 4), rot=0)
        ax.set_title(f"Experiment 4 – {label} (beide Datensätze)")
        ax.set_ylabel(label)
        ax.set_xlabel("")
        ax.legend(title="Datensatz")
        plt.tight_layout()
        plt.show()

## Experiment 5 – TFM-Erklärbarkeit
- Feature-Attributionen für einen TabPFN-Klassifikator: ShapPFN & ExplainerPFN vs. KernelSHAP (Baseline)
- Je Explainer: Fidelity vs. KernelSHAP (Spearman + Cosine) und Laufzeit; KernelSHAP selbst = Referenz

In [ ]:
for ds in datasets:
    sub = df[(df["exp_num"] == 5) & (df["dataset"] == ds)] if len(df) else df
    if not len(sub):
        print(f"{ds}: keine Exp-5-Runs")
        continue
    runs_total = sub.groupby("model").size()
    latest = sub.sort_values("start_time").groupby("model").tail(1).set_index("model")  # nur der neueste Run je Modell
    cols = {"metrics.pearson_vs_kernelshap": "pearson_vs_KernelSHAP",
            "metrics.spearman_vs_kernelshap": "spearman_vs_KernelSHAP",
            "metrics.cosine_vs_kernelshap": "cosine_vs_KernelSHAP",
            "runtime_s": "runtime_s"}
    tab = pd.DataFrame(index=latest.index)
    for src, dst in cols.items():
        tab[dst] = latest[src] if src in latest.columns else np.nan
    tab["runs_total"] = runs_total
    # KernelSHAP (Referenz) zuerst, dann nach Fidelity absteigend
    sp = tab["spearman_vs_KernelSHAP"]
    order = sorted(tab.index, key=lambda m: (m != "kernelshap", -(sp[m] if pd.notna(sp[m]) else -np.inf)))
    tab = tab.loc[order].round(4)
    tab.to_csv(f"{RESULT_DIR}/exp5_{ds}.csv")
    print(f"=== Experiment 5 – {ds}: Explainer-Fidelity vs. KernelSHAP (neuester Run) ===")
    display(tab)
    fid = tab["spearman_vs_KernelSHAP"].dropna()
    if len(fid):
        ax = fid.plot.bar(figsize=(7, 4), rot=0)
        ax.set_title(f"Experiment 5 – {ds}: Fidelity (Spearman vs. KernelSHAP)")
        ax.set_ylabel("Spearman")
        ax.set_xlabel("")
        ax.axhline(0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()